# Jev basics --- Noul, Choice, Score

The **hello world** for [Jev](https://docs.typesafe.ai/), TypeSafe's *System One*
classifier model. No agent, no Strands --- just the model.

Jev doesn't generate text. You send it a **state** (the thing to judge) plus one or
more **typed questions**, and it returns **typed answers** with probabilities, all in
one parallel call. There are three question types:

| Type | Answers | Returns |
|------|---------|---------|
| **Noul** | Is this statement true? | `.noul` --- a probability 0..1 |
| **Choice** | Which one of these options? | `.choice`, `.confidence`, `.probabilities` |
| **Score** | Which level on this scale? | `.score`, `.confidence`, `.legend` |

The point to take away: every answer is **typed and constrained to what you asked
for**. There's no prose to parse and nothing comes back off-schema. Your code reads
`.noul` / `.choice` / `.score` directly and decides what to do.

We'll look at each type on its own, then combine all three in one call.

## Setup

Install the SDK and load your TypeSafe key. Create a `.env` file next to this
notebook with:

```
TYPESAFE_API_KEY=your-key-here
```

In [1]:
%pip install -q typesafe-sdk python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

load_dotenv()  # reads TYPESAFE_API_KEY from .env

jev = TypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"])
print("client ready")

client ready


## The shared state

Every question below judges this same support message. In a real system the state
could be a ticket, a chat transcript, a document --- any content you want a judgment
about.

In [3]:
STATE = (
    "I've been trying to connect my Stripe account for three days and the "
    "integration keeps failing. I'm losing sales. Please help ASAP."
)
print(STATE)

I've been trying to connect my Stripe account for three days and the integration keeps failing. I'm losing sales. Please help ASAP.


## 1. Noul --- is this statement true?

Use a **Noul** for a clean yes/no question where the probability *itself* is the
signal: "does this contain PII", "is the customer asking for a refund", "is this
urgent".

Phrase the statement so a **high** value means *yes*. A Noul has **no separate
confidence** --- its distance from 0.5 *is* the decisiveness: `0.97` is a firm yes,
`0.03` a firm no, `0.5` means "no idea".

In [4]:
answer = jev.system_one(
    state=STATE,
    questions={
        "is_urgent": Noul(
            instructions="The message conveys urgency or time-sensitivity",
        ),
    },
).answers["is_urgent"]

print(f"noul = {answer.noul:.2f}   (probability the statement is true)")

noul = 1.00   (probability the statement is true)


`.noul` is the whole answer --- one number your code can threshold on, e.g.
`if answer.noul > 0.5:`. No parsing, no "maybe" buried in a sentence.

## 2. Choice --- pick one option from a fixed set

Use a **Choice** when the answer is one of a known list with no order between them:
routing a ticket, classifying a document, detecting a language.

Jev can **only** ever return one of the options you supply --- it can't go
off-schema. You also get the full probability distribution and a confidence.

In [5]:
answer = jev.system_one(
    state=STATE,
    questions={
        "department": Choice(
            instructions="Which team should handle this message",
            criteria={
                "billing": "Payments, invoices, or subscription issues",
                "technical": "Bugs or integration problems",
                "sales": "Pricing or account questions",
            },
        ),
    },
).answers["department"]

print(f"choice        = {answer.choice}")
print(f"confidence    = {answer.confidence:.2f}")
print(f"probabilities = {answer.probabilities}")

choice        = technical
confidence    = 0.68
probabilities = {'technical': 0.79, 'billing': 0.21, 'sales': 0.0}


`.choice` is the winning option; `.probabilities` tells you *by how much*. A
`0.76 / 0.24` split and a `0.41 / 0.38` split can pick the same option but mean very
different things --- read the spread before you auto-act. `.confidence` collapses
that spread into one number you can threshold on.

## 3. Score --- rate on an ordered scale

Use a **Score** when the answer sits on a spectrum you can describe level by level:
severity, frustration, completeness, skill.

Give 2..10 levels, **lowest first**. The returned score can land *between* levels ---
treat it as a **rank, not a measurement** (a `1.5` doesn't mean "5 hours"; it means
"between level 1 and 2").

In [6]:
answer = jev.system_one(
    state=STATE,
    questions={
        "frustration": Score(
            instructions="How frustrated does the customer sound",
            criteria=[
                "Calm, just stating facts",
                "Frustrated but civil",
                "Very angry, strong language",
            ],
        ),
    },
).answers["frustration"]

print(f"score      = {answer.score:.2f}   (0=calm .. 2=very angry)")
print(f"confidence = {answer.confidence:.2f}")
print(f"legend     = {answer.legend}")

score      = 1.00   (0=calm .. 2=very angry)
confidence = 1.00
legend     = {0: 'Calm, just stating facts', 1: 'Frustrated but civil', 2: 'Very angry, strong language'}


`.score` is a position along *your* levels, `.legend` maps each number back to
the level you described, and `.confidence` says how peaked the distribution is.

## 4. All three at once --- then let plain Python decide

Every question in a request sees the **same state**, runs **in parallel**, and comes
back under the ID you chose. Asking more questions barely changes latency, so ask
everything you might need in one call and let your code combine the answers.

This is the key pattern: **Jev tells you *what is*; your code decides *what
happens*.** The policy below is ordinary, readable Python --- change it without
touching the model.

In [7]:
answers = jev.system_one(
    state=STATE,
    questions={
        "department": Choice(
            instructions="Which team should handle this message",
            criteria={
                "billing": "Payments, invoices, or subscription issues",
                "technical": "Bugs or integration problems",
                "sales": "Pricing or account questions",
            },
        ),
        "frustration": Score(
            instructions="How frustrated does the customer sound",
            criteria=[
                "Calm, just stating facts",
                "Frustrated but civil",
                "Very angry, strong language",
            ],
        ),
        "is_urgent": Noul(instructions="The message conveys urgency"),
    },
).answers

department = answers["department"].choice
frustration = answers["frustration"].score
urgent = answers["is_urgent"].noul

# The policy is plain Python. Jev classified; we decide.
priority = "high" if (urgent > 0.5 or frustration >= 1.5) else "normal"

print(f"department  : {department}")
print(f"frustration : {frustration:.2f}")
print(f"is_urgent   : {urgent:.2f}")
print(f"\n-> route to {department!r} at {priority!r} priority")

department  : technical
frustration : 1.00
is_urgent   : 1.00

-> route to 'technical' at 'high' priority


## Recap

- **Noul** --- yes/no probability; threshold on distance from 0.5.
- **Choice** --- one of a fixed set; read `.choice` and the `.probabilities` spread.
- **Score** --- a rank along your levels; use `.score` against a threshold.
- **Combine** them in one call and turn the typed answers into a decision with plain
  Python.

Next: see `demos/escalation_gate/main.py` for these primitives inside a real Strands +
Claude agent that decides when to hand a live call to a human.